# load orders data to stage table 

In [0]:
source_dir = '/Volumes/event_driven_catalog/default/event_driven_file_store/orders_data/source/'
archive_dir = '/Volumes/event_driven_catalog/default/event_driven_file_store/orders_data/source/'
stage_table = 'event_driven_catalog.default.orders_stage'
error_table = 'event_driven_catalog.default.orders_error'

print("Loading orders data from {source_dir} into {stage_table} ")

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, StringType, IntegerType ,DateType, DecimalType , TimestampType

defined_schema = StructType([
    StructField('order_id',StringType(),nullable=True),
    StructField('customer_id',IntegerType(),nullable=True),
    StructField('product_id',StringType(),nullable=True),
    StructField('order_date',DateType(),nullable=True),
    StructField('order_amount',DecimalType(),nullable=True),
    StructField('currency',StringType(),nullable=True),
    StructField('payment_method',StringType(),nullable=True),
    StructField('shipping_address',StringType(),nullable=True),
    StructField('order_status',StringType(),nullable=True),
    StructField('created_timestamp',TimestampType(),nullable=True)
])

print("Schema defined for Orders")


In [0]:
from datetime import datetime

try:
    df = spark.read.schema(defined_schema).csv(
        source_dir,
        header=True,
        dateFormat='yyyy-MM-dd',
        timestampFormat='yyyy-MM-dd HH:mm:ss'
    )

    # Metadata
    df = (df.withColumn("Processed_Timestamp", f.current_timestamp())
            .withColumn("batch_id", f.lit(datetime.now().strftime("%Y%m%d%H%M%S")))
            .withColumn("source_system", f.lit("ecommence_orders")))

    # Data quality checks
    total_records = df.count()
    null_orders_count = df.filter(f.col("order_id").isNull()).count()
    null_customer_count = df.filter(f.col("customer_id").isNull()).count()
    invalid_amount = df.filter(f.col("order_amount") < f.lit(0))

    print(f"Total records in the file: {total_records}")
    print(f"Null order_id count: {null_orders_count}")
    print(f"Null customer_id count: {null_customer_count}")
    print(f"Invalid amount count: {invalid_amount.count()}")

    df_valid = df.filter(
        f.col("order_id").isNotNull() &
        f.col("customer_id").isNotNull() &
        (f.col("order_amount") >= f.lit(0))
    )

    df_invalid = df.filter(
        f.col("order_id").isNull() |
        f.col("customer_id").isNull() |
        (f.col("order_amount") < f.lit(0))
    )

    print(f"Valid records count: {df_valid.count()}")
    print(f"Invalid records count: {df_invalid.count()}")

except Exception as e:
    print(f"An error occurred while processing the source file: {e}")
    raise